### Storing in a vector store
For this application I choose the FAISS vector db. Reasons are:
- It is light weight : As per the instructions we have to make a light weight semantic search engine so FAISS seemed to be the best choice. 
- Other choices like chromadb or pinecone have heavier dependencies and little over kill for this partiular application

<br>
<br>
For retrieval I am using cosine similarity. FAISS internally uses inner product.To make them equivalent I have normalized the embeddings.

In [ ]:
import faiss
import numpy as np

def build_faiss_index(embeddings):
    """
    take the embeddings already generated normalize them and return the index
    """

    embeddings = np.array(embeddings).astype("float32")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)

    # normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)

    index.add(embeddings)

    return index

In [ ]:
embeddings = np.load("embeddings.npy")
index = build_faiss_index(embeddings)
print("Total vectors:", index.ntotal)

Total vectors: 19997


In [3]:
faiss.write_index(index, "newsgroups.index")

In [ ]:
def load_index(path="newsgroups.index"):
    return faiss.read_index(path)

In [ ]:
def search(index, query_embedding, k=5):
    query_embedding = np.array([query_embedding]).astype("float32")
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, k)
    return scores[0], indices[0]

In [9]:
scores, indices = search(index, embeddings[6789], k=5)

print(indices)
print(scores)

[6789 8095 8231 8321 8952]
[0.99999994 0.56827855 0.5232373  0.50443834 0.5015219 ]


In [11]:
from sentence_transformers import SentenceTransformer
import pandas as pd

model = SentenceTransformer("all-MiniLM-L6-v2")
df = pd.read_csv("vectorized_data.csv")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
query = "space shuttle mission"
query_embedding = model.encode([query])[0]
scores, indices = search(index, query_embedding)
print(scores)
print(df.iloc[indices]["clean_text"])

[0.5281789  0.5216878  0.52036333 0.5105526  0.492389  ]
14344    HST Servicing Mission Scheduled for 11 Days. E...
14841    HST Servicing Mission Scheduled for 11 Days. I...
14599    Jemison on Star Trek (Better Ideas). In articl...
14010    Space FAQ 09/15 - Mission Schedules. Archive-n...
14642    Historic shuttle flights. Would someone please...
Name: clean_text, dtype: str
